- **Import dependencies**

In [63]:
import numpy as np
import pandas as pd
import glob

- **Import data**

In [75]:
# Import files globally
files = glob.glob('/Users/Euan Bronsky/Downloads/THESIS DATA/data*/*')

# Read the files and concatenate into a dataframe
data = pd.concat([pd.read_csv(file, sep = ',', engine = 'c', low_memory=False) for file in files], ignore_index=True)

- **Prepare data for cleaning**

In [ ]:
# Create list of relevant columns
relevant_cols = ['quote_date', 'underlying_last', 'expire_date', 'dte', 'c_iv', 'c_bid', 'c_ask', 'c_delta', 'strike', 'p_bid', 'p_ask', 'p_iv', 'p_delta']
numeric_cols = ['underlying_last', 'dte', 'c_iv', 'c_bid', 'c_ask', 'c_delta', 'strike', 'p_bid', 'p_ask', 'p_iv', 'p_delta']
date_cols = ['quote_date', 'expire_date']
common_cols = ['quote_date', 'underlying_last', 'expire_date', 'dte', 'strike']

# Adjust columns and keep only relevant columns
data.columns = data.columns.str.lower().str.replace(r'[\[\]\s]','', regex=True)
df = data[relevant_cols].copy()

# Convert into numerical and date values
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')


In [76]:
# Separate calls from puts
calls = df[common_cols + ['c_iv', 'c_bid', 'c_ask', 'c_delta']].copy()
calls = calls.rename(columns={'c_bid': 'bid', 'c_ask': 'ask', 'c_iv': 'iv', 'c_delta': 'delta'})
calls['option_type'] = 'call'

# Separate puts from calls
puts = df[common_cols + ['p_iv', 'p_bid', 'p_ask', 'p_delta']].copy()
puts = puts.rename(columns={'p_bid': 'bid', 'p_ask': 'ask', 'p_iv': 'iv', 'p_delta': 'delta'})
puts['option_type'] = 'put'

# Concatenate the results into a long format instead of wide
df = pd.concat([calls, puts], ignore_index=True)

# Sort data by option type, quote date, expiry date, and strike
df = df.sort_values(by = ['option_type', 'quote_date', 'expire_date', 'strike'])

- **Filtering procedure**

In [97]:
def cleaning_fun(df):

    df = df.copy()

    # Step 1: compute mid prices
    df['mid'] = (df['bid'] + df['ask'])/2

    # Step 2
    df1 = df[df['mid'] >= 3/8]

    # Step 3: less than one week TTM removal
    df2 = df1[(df1['dte'] >= 7) & (df1['dte'] <= 360)]

    # Step 4: removal of ITM options
    df3 = df2[((df2['option_type'] == 'call') & (df2['strike'] >= 0.97 * df2['underlying_last'])) | ((df2['option_type'] == 'put') & (df2['strike'] <= 1.03 * df2['underlying_last']))]

    return df3

- **Arbitrage filter**

In [101]:
def arbitrage_filter(df3, r):

    df3 = df3.copy()

    # Step 1a: Put and call lower bounds
    df3 = df3[((df3['option_type'] == 'call') & (df3['mid'] >= np.maximum(df3['underlying_last'] - df3['strike']*np.exp(-r*df3['dte']/365),0)) | (df3['option_type'] == 'put') & (df3['mid'] >= np.maximum(df3['strike']*np.exp(-r*df3['dte']/365) - df3['underlying_last'], 0)))]

    # Step 1b: Put and call upper bounds
    df4 = df3[((df3['option_type'] == 'call') & (df3['mid'] <= df3['underlying_last'])) | (df3['option_type'] == 'put') & (df3['mid'] <= df3['strike']*np.exp(-r*df3['dte']/365))]

    # Sort by strike
    df4 = df4.sort_values(['option_type', 'quote_date', 'expire_date', 'strike'])

    # Step 2: Monotonicity in strike
    diffs = df4.groupby(['option_type', 'quote_date', 'expire_date'])['mid'].diff()
    df5 = df4[((df4['option_type'] == 'call') & (diffs.isna() | (diffs <= 0))) | ((df4['option_type'] == 'put') & (diffs.isna() | (diffs >= 0)))]

    # Step 3: Strike convexity
    slopes = (df5.groupby(['option_type', 'quote_date', 'expire_date'])['mid'].diff() / df5.groupby(['option_type', 'quote_date', 'expire_date'])['strike'].diff())
    slope_diffs = slopes.groupby([df5['option_type'], df5['quote_date'], df5['expire_date']]).diff()
    df6 = df5[slope_diffs.isna() | (slope_diffs >=0)]

    # Step 4: Calendar monotonicity
    df6 = df6.sort_values(['option_type', 'quote_date', 'strike', 'dte'])
    diffs_ttm = df6.groupby(['option_type', 'quote_date', 'strike'])['mid'].diff()
    df7 = df6[diffs_ttm.isna() | (diffs_ttm >= 0)]

    return df7

31112954